In [1]:
import csv, sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'

con = sqlite3.connect("q.db")
cur = con.cursor()

In [2]:
%load_ext sql
%sql sqlite:///q.db

In [3]:
import pandas as pd
df = pd.read_csv("spacex_launch_dash.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

56

In [4]:
%sql DROP TABLE IF EXISTS SPACEXTABLE;

 * sqlite:///q.db
Done.


[]

In [7]:
%sql create table SPACEXTABLE as select * from SPACEXTBL

 * sqlite:///q.db
Done.


[]

In [9]:
%sql SELECT * FROM SPACEXTABLE WHERE class = 1 LIMIT 3;

 * sqlite:///q.db
Done.


Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
17,19,CCAFS LC-40,1,1952.0,F9 v1.1 B1018,v1.1
18,20,CCAFS LC-40,1,2034.0,F9 FT B1019,FT
20,23,CCAFS LC-40,1,3136.0,F9 FT B1021.1,FT


Which site has the largest successful launches?

VAFB SLC-4E


In [19]:
#%sql SELECT "Launch Site" class "Payload Mass (kg)"
%sql SELECT "Launch Site", class, "Payload Mass (kg)"\
FROM SPACEXTABLE\
WHERE class = 1\
ORDER BY "Payload Mass (kg)" DESC;

 * sqlite:///q.db
Done.


Launch Site,class,Payload Mass (kg)
VAFB SLC-4E,1,9600.0
VAFB SLC-4E,1,9600.0
VAFB SLC-4E,1,9600.0
KSC LC-39A,1,5300.0
KSC LC-39A,1,5200.0
KSC LC-39A,1,4990.0
CCAFS LC-40,1,4696.0
CCAFS LC-40,1,4600.0
KSC LC-39A,1,3696.65
CCAFS SLC-40,1,3696.65


Which site has the highest launch success rate?

KSC LC-39A

In [21]:
%sql SELECT "Launch Site", AVG(class) as success\
FROM SPACEXTABLE\
GROUP BY "Launch Site";

 * sqlite:///q.db
Done.


Launch Site,success
CCAFS LC-40,0.2692307692307692
CCAFS SLC-40,0.42857142857142855
KSC LC-39A,0.7692307692307693
VAFB SLC-4E,0.4


Which payload range(s) has the highest launch success rate?

3000 - 4000 KG

In [26]:
%sql SELECT CEILING("Payload Mass (kg)" / 1000) as weight, AVG(class) as success\
FROM SPACEXTABLE\
GROUP BY weight\
ORDER BY success;

 * sqlite:///q.db
Done.


weight,success
0.0,0.0
7.0,0.0
1.0,0.25
2.0,0.3333333333333333
5.0,0.375
6.0,0.4
3.0,0.5
10.0,0.6
4.0,0.7272727272727273


Which payload range(s) has the lowest launch success rate?

0 - 1000 KG

Which F9 Booster version (v1.0, v1.1, FT, B4, B5, etc.) has the highest launch success rate?

In [31]:
%sql SELECT SUBSTRING("Booster Version", 3, 5) as version, AVG(class) as success\
FROM SPACEXTABLE\
GROUP BY version\
ORDER BY success;

 * sqlite:///q.db
Done.


version,success
B4,0.0
v1.0,0.0
v1.1,0.06666666666666667
FT,0.5714285714285714
FT B,0.7058823529411765
B4 B,0.8571428571428571
B5,1.0


In [32]:
%sql SELECT SUBSTRING("Booster Version", 3, 5) as version,\
COUNT(SUBSTRING("Booster Version", 3, 5)) as count, AVG(class) as success\
FROM SPACEXTABLE\
GROUP BY version\
ORDER BY success;

 * sqlite:///q.db
Done.


version,count,success
B4,4,0.0
v1.0,5,0.0
v1.1,15,0.06666666666666667
FT,7,0.5714285714285714
FT B,17,0.7058823529411765
B4 B,7,0.8571428571428571
B5,1,1.0
